# Project 3: Sales Forecasting
## Notebook 7: Dataset Switcher — Train on Any of the 12 Datasets

**How to use this notebook:**
1. Pick a dataset key in Step 1
2. Download the CSV from Kaggle and drop it in `data/raw/`
3. Run all cells — the same pipeline retrains on the new data automatically

**The code doesn't change. Only the dataset key changes.**
That's the whole point — one reusable forecasting engine, many datasets.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

BASE = Path(r'C:\Users\Administrator\SalesForecast')
sys.path.insert(0, str(BASE / 'src'))

from dataset_loader import load_dataset, list_datasets, DATASETS
from feature_engineering import create_ml_features, compute_business_insights

PROC_DIR  = BASE / 'data' / 'processed'
MODEL_DIR = BASE / 'models'
FIG_DIR   = BASE / 'reports' / 'figures'
for d in [PROC_DIR, MODEL_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## Step 1: Choose Your Dataset

Change `DATASET_KEY` to any key from the table below.

In [ ]:
list_datasets()

In [ ]:
# ── CHANGE THIS LINE TO SWITCH DATASET ───────────────────────────────────
DATASET_KEY = 'kenyan_fmcg'   # ← try: 'retail_store', 'fmcg_profit', 'east_africa'
# ─────────────────────────────────────────────────────────────────────────

meta = DATASETS[DATASET_KEY]
print(f"Selected: {meta['name']}")
print(f"Industry: {meta['industry']}")
print(f"Rows:     {meta['rows']}")
print(f"Description: {meta['description']}")
print(f"\nKaggle: {meta['kaggle']}")
print(f"Expected files in data/raw/:")
for f in meta['files']:
    p = BASE / 'data' / 'raw' / f
    status = '✅ Found' if p.exists() else '❌ Missing — download from Kaggle'
    print(f"  {f} — {status}")

## Step 2: Load & Inspect

In [ ]:
monthly, cat_monthly, df_raw = load_dataset(DATASET_KEY)

print(f'\n=== Dataset Summary ===')
print(f'Monthly data points: {len(monthly)}')
print(f'Categories: {list(cat_monthly.columns)}')
print(f'Date range: {monthly.index[0].date()} → {monthly.index[-1].date()}')
print(f'\nWARNING: Need at least 24 months for SARIMA decomposition.')
if len(monthly) < 24:
    print(f'  Only {len(monthly)} months available — SARIMA will be skipped, Prophet + LightGBM will run.')

monthly.tail()

In [ ]:
# Quick EDA
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Sales trend
axes[0,0].plot(monthly.index, monthly.values, color='#2E75B6', lw=2)
axes[0,0].fill_between(monthly.index, monthly.values, alpha=0.15, color='#2E75B6')
axes[0,0].set(title=f'{meta["name"]} — Monthly Sales', ylabel='Sales')

# Rolling average
roll3  = monthly.rolling(3, min_periods=1).mean()
roll12 = monthly.rolling(12, min_periods=1).mean()
axes[0,1].plot(monthly.index, monthly.values, color='lightgray', lw=1, label='Monthly')
axes[0,1].plot(roll3.index,  roll3.values,  color='#2E75B6', lw=2, label='3-mo MA')
axes[0,1].plot(roll12.index, roll12.values, color='#DD8452', lw=2, label='12-mo MA')
axes[0,1].set(title='Rolling Averages')
axes[0,1].legend(fontsize=8)

# By category
if not cat_monthly.empty and len(cat_monthly.columns) > 1:
    for col in cat_monthly.columns[:5]:  # max 5 categories
        axes[1,0].plot(cat_monthly.index, cat_monthly[col], lw=2, label=col)
    axes[1,0].set(title='Sales by Category')
    axes[1,0].legend(fontsize=7)

# Monthly distribution
month_avg = monthly.groupby(monthly.index.month).mean()
mnames = ['J','F','M','A','M','J','J','A','S','O','N','D']
axes[1,1].bar(range(1,13), [month_avg.get(i,0) for i in range(1,13)],
              color=['#C44E52' if m in [10,11,12] else '#2E75B6' for m in range(1,13)],
              edgecolor='white')
axes[1,1].set(xticks=range(1,13), xticklabels=mnames, title='Avg Sales by Month')

plt.suptitle(f'EDA — {meta["name"]}', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / f'eda_{DATASET_KEY}.png', bbox_inches='tight')
plt.show()

## Step 3: Save Processed Data

Overwrites the processed data so all other notebooks use the new dataset.

In [ ]:
monthly.to_csv(PROC_DIR / 'monthly_sales.csv')
cat_monthly.to_csv(PROC_DIR / 'monthly_sales_by_category.csv')

# Save dataset metadata so dashboard knows which dataset is active
with open(PROC_DIR / 'active_dataset.json', 'w') as f:
    json.dump({
        'key':         DATASET_KEY,
        'name':        meta['name'],
        'industry':    meta['industry'],
        'n_months':    len(monthly),
        'date_range':  f"{monthly.index[0].date()} → {monthly.index[-1].date()}",
        'categories':  list(cat_monthly.columns),
    }, f, indent=2)

print(f'Saved processed data for: {meta["name"]}')
print(f'  data/processed/monthly_sales.csv')
print(f'  data/processed/monthly_sales_by_category.csv')
print(f'  data/processed/active_dataset.json')

## Step 4: Quick Prophet Forecast

Fast check — does the new dataset produce a sensible forecast?

In [ ]:
from prophet import Prophet

TEST_MONTHS = min(12, len(monthly) // 4)  # use 25% for test, max 12
HORIZON     = 12

train_df = pd.DataFrame({'ds': monthly.index[:-TEST_MONTHS], 'y': monthly.values[:-TEST_MONTHS]})
test_df  = pd.DataFrame({'ds': monthly.index[-TEST_MONTHS:],  'y': monthly.values[-TEST_MONTHS:]})

m = Prophet(
    yearly_seasonality=True if len(monthly) >= 24 else False,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    interval_width=0.95,
)
m.fit(train_df)

future   = m.make_future_dataframe(periods=TEST_MONTHS, freq='MS')
forecast = m.predict(future)
test_fc  = forecast.tail(TEST_MONTHS)

y_true = test_df['y'].values
y_pred = test_fc['yhat'].values
mape   = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
rmse   = np.sqrt(np.mean((y_true - y_pred)**2))

print(f'Prophet on {meta["name"]}')
print(f'  MAPE: {mape:.2f}%')
print(f'  RMSE: {rmse:,.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_df['ds'], train_df['y'], color='#2E75B6', lw=2, label='Training')
ax.plot(test_df['ds'],  test_df['y'],  color='#2E75B6', lw=2, linestyle='--', label='Actual')
ax.plot(test_fc['ds'],  test_fc['yhat'], color='#55A868', lw=2.5, label=f'Prophet (MAPE={mape:.1f}%)')
ax.fill_between(test_fc['ds'], test_fc['yhat_lower'], test_fc['yhat_upper'],
                alpha=0.20, color='#55A868', label='95% CI')
ax.axvline(test_df['ds'].iloc[0], color='gray', linestyle=':', alpha=0.7)
ax.set(title=f'Prophet Forecast — {meta["name"]}', ylabel='Sales')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / f'forecast_{DATASET_KEY}.png', bbox_inches='tight')
plt.show()

print(f'\n✅  Dataset {DATASET_KEY} ready.')
print('Now run notebooks 02–06 to retrain all models on this dataset.')
print('The dashboard will automatically use the new data.')

## Step 5: Business Insights for This Dataset

In [ ]:
insights = compute_business_insights(df_raw.copy())

print(f'=== Business Insights: {meta["name"]} ===')
for k, v in insights.items():
    print(f'  {k}: {v}')

## Dataset Comparison: Which Should You Use?

| Goal | Best Dataset |
|---|---|
| Quick demo / default | `superstore` |
| Profit & marketing ROI | `fmcg_profit` |
| Multi-store with economic data | `retail_store` |
| Large-scale ML (1.2M rows) | `synthetic_retail` |
| Stand out on African platforms | `kenyan_fmcg` or `east_africa` |
| B2B / enterprise sales | `b2b_crm` |
| Global economic impact | `ecommerce_multi` |
| Small quick test | `fmcg_demand` |